# X05 Knowledge Neurons


## Goal

Localizes 'knowledge' into a more intervention-friendly set of activations and offers another classic route from feature readout toward local editing and attribution.


## Why this paper matters now

This paper forces you to separate 'some units predict the phenomenon' from 'those units truly carry the phenomenon.'


## Notebook and deliverable

- Source: https://arxiv.org/abs/2104.08696
- Notebook: `notebooks/extensions/en/x05_knowledge_neurons.ipynb`
- Prereqs: X04, M04
- Reproduce a teaching-scale knowledge-neuron scoring and ablation experiment in Colab, then write one note explaining why a high-scoring neuron still does not automatically yield a causal explanation.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import torch
from torch import nn

prompts = torch.eye(4)
probe = prompts[0:1].clone().requires_grad_(True)
model = nn.Sequential(
    nn.Linear(4, 6),
    nn.ReLU(),
    nn.Linear(6, 4),
)
with torch.no_grad():
    model[0].weight.zero_()
    model[0].bias.zero_()
    model[0].weight[0, 0] = 4.0
    model[0].weight[1, 1] = 4.0
    model[0].weight[2, 2] = 4.0
    model[0].weight[3, 3] = 4.0
    model[2].weight.zero_()
    model[2].bias.zero_()
    model[2].weight[0, 0] = 2.5
    model[2].weight[1, 1] = 2.5
    model[2].weight[2, 2] = 2.5
    model[2].weight[3, 3] = 2.5
    model[2].weight[0, 4] = 0.4
    model[2].weight[1, 5] = 0.4

hidden = model[1](model[0](probe))
logits = model[2](hidden)
target_logit = logits[0, 0]
grads = torch.autograd.grad(target_logit, hidden)[0]
scores = (hidden * grads).detach().flatten()
top_idx = int(scores.abs().argmax())

with torch.no_grad():
    original = model(prompts[0:1]).softmax(dim=-1).flatten()
    hidden_ablated = hidden.detach().clone()
    hidden_ablated[0, top_idx] = 0.0
    ablated = model[2](hidden_ablated).softmax(dim=-1).flatten()

print("Knowledge-neuron scores:", [round(float(x), 4) for x in scores])
print("Top-scoring hidden unit:", top_idx)
print("Original distribution:", [round(float(x), 4) for x in original])
print("After ablating top unit:", [round(float(x), 4) for x in ablated])


## Takeaway

A high-scoring neuron is evidence worth testing, not a free pass to a causal story.


## Colab-first replication workflow


### Before you run

- Write one prediction about the mechanism before you run the cells.
- Name the baseline you are comparing against.
- Decide what result would count as a failure of your favorite story.


### After you run

- Separate observation from inference in your notes.
- Mark one ambiguity that still remains after the reproduction.
- Write one next experiment that would reduce that ambiguity.


### Ship these outputs

- Reproduce a teaching-scale knowledge-neuron scoring and ablation experiment in Colab, then write one note explaining why a high-scoring neuron still does not automatically yield a causal explanation.
- One experiment log with the exact settings you changed.
- One paragraph called 'what this reproduction still does not prove'.


## Self-check questions

- Why can a high neuron score still mark correlation rather than a storage site?
- In your toy ablation, what outcome most supports a 'local carrier' interpretation?
- What is the biggest difference in explanatory strength between knowledge neurons and feature probes?
- If you cannot answer at least two questions without reopening the notebook, rerun the reproduction and rewrite the memo.
